# Breaking Hardware AES on XMEGA MCU (and other FPGA targets).

This tutorial relies on previous knowledge from the SCA101 course notebooks (in `../courses/sca101/`); make sure you go through these first to understand how a CPA attack works.

In this notebook, we'll apply knowledge from sca101 to break a hardware AES implementation on the CW305 Artix FPGA.

Some out-of-date background on the target FPGA project is can be found here: [Tutorial CW305-1 Building a Project](http://wiki.newae.com/Tutorial_CW305-1_Building_a_Project) (ignore the "capture setup" section, which uses the obsolete ChipWhisperer GUI; this notebook shows all you need to know about capture setup on the CW305 with Jupyter).

While this notebook has "CW305" in its title, it can also be used with the CW312T-A35 and CW312T-iCE40 targets.

## Background Theory
During this tutorial, we'll be working with a hardware AES implementation. This type of attack can be much more difficult than a software AES attack. In the software AES attacks, we needed hundreds or thousands of clock cycles to capture the algorithm's full execution. In contrast, a hardware AES implementation may have a variety of speeds. Depending on the performance of the hardware, a whole spectrum of execution speeds can be achieved by executing many operations in a single clock cycle. It is theoretically possible to execute the entire AES encryption in a single cycle, given enough hardware space and provided that the clock is not too fast. Most hardware accelerators are designed to complete one round or one large part of a round in a single cycle.

This fast execution may cause problems with a regular CPA attack. In software, we found that it was easy to search for the outputs of the s-boxes because these values would need to be loaded from memory onto a high-capacitance data bus. This is not necessarily true on an FPGA, where the output of the s-boxes may be directly fed into the next stage of the algorithm. In general, we may need some more knowledge of the hardware implementation to successfully complete an attack.

In our case, let's suppose that every round of AES is completed in a single clock cycle. Recall the execution of AES:

<img src="img/aes_operations.png" width="250">

Here, every blue block is executed in one clock cycle. This means that an excellent candidate for a CPA attack is the difference between the input and output of the final round. It is likely that this state is stored in a port that is updated every round, so we expect that the Hamming distance between the round input and output is the most important factor on the power consumption. Also, the last round is the easiest to attack because it has no MixColumns operation. We'll use this Hamming distance as the target in our CPA attack.

## Capture Notes

Most of the capture settings used below are similar to the standard ChipWhisperer scope settings. However, there are a couple of interesting points:

- We're only capturing 129 samples (the minimum allowed with CW-lite), and the encryption is completed in less than 60 samples with an x4 ADC clock. This makes sense - as we mentioned above, our AES implementation is computing each round in a single clock cycle.
- We're using EXTCLK x4 for our ADC clock. This means that the FPGA is outputting a clock signal, and we aren't driving it.

Other than these, the last interesting setting is the number of traces. By default, the capture software is ready to capture 5000 traces - many more than were required for software AES! It is difficult for us to measure the small power spikes from the Hamming distance on the last round: these signals are dwarfed by noise and the other operations on the chip. To deal with this small signal level, we need to capture many more traces.

## Capture Setup

Setup is somewhat similar to other targets, except that we are using an external clock (driven from the FPGA-- unless you're using the CW312T-A35 target). We'll also do the rest of the setup manually:

In [ ]:
!cd ../../firmware/mcu/simpleserial-aes && make PLATFORM=CWLITEXMEGA CRYPTO_TARGET=TINYAES128C

In [ ]:
import chipwhisperer as cw
import time
import os

scope = cw.scope()
scope.default_setup()

# XMEGA target setup
scope.gain.db = 25
scope.adc.samples = 3000
scope.adc.offset = 0
scope.adc.basic_mode = "rising_edge"

# Standard SimpleSerial/XMEGA routing
scope.clock.clkgen_freq = 7.37e6
scope.clock.adc_src = "clkgen_x4"

scope.trigger.triggers = "tio4"
scope.io.tio1 = "serial_rx"
scope.io.tio2 = "serial_tx"
scope.io.hs2 = "clkgen"

time.sleep(0.1)

# Ensure ADC lock
for _ in range(5):
    scope.clock.reset_adc()
    time.sleep(0.5)
    if scope.clock.adc_locked:
        break

assert scope.clock.adc_locked, "ADC failed to lock"
print("Scope setup complete.")

In [ ]:
from pathlib import Path

FW_PATH = Path("../../firmware/mcu/simpleserial-aes/simpleserial-aes-CWLITEXMEGA.hex").resolve()

print("Using firmware:", FW_PATH)
assert FW_PATH.exists(), f"Firmware not found: {FW_PATH}"

In [ ]:
prog = cw.programmers.XMEGAProgrammer
cw.program_target(scope, prog, str(FW_PATH))

print("XMEGA programmed successfully.")

In [ ]:
target = cw.target(scope, cw.targets.SimpleSerial)
target.flush()

print("Connected to XMEGA SimpleSerial target.")

Next we set all the PLLs. We enable CW305's PLL1; this clock will feed both the target and the CW ADC. As explained [here](http://wiki.newae.com/Tutorial_CW305-1_Building_a_Project#Capture_Setup), **make sure the DIP switches on the CW305 board are set as follows**:
- J16 = 0
- K16 = 1

In [ ]:
ktp = cw.ktp.Basic()
key, text = ktp.next()

ret = cw.capture_trace(scope, target, text, key)

assert ret is not None, "Capture failed"
print("Trace length:", len(ret.wave))
print("Plaintext :", bytes(text).hex())
print("Key       :", bytes(key).hex())
print("Ciphertext:", bytes(ret.textout).hex())

Occasionally the ADC will fail to lock on the first try; when that happens, the above assertion will fail (and on the CW-Lite, the red LED will be on). Simply re-running the above cell again should fix things.

## Trace Capture
Below is the capture loop. The main body of the loop loads some new plaintext, arms the scope, sends the key and plaintext, then finally records and appends our new trace to the `traces[]` list.

Because we're capturing 5000 traces, this takes a bit longer than the attacks against software AES implementations.

Note that the encryption result is read from the target and compared to the expected results, as a sanity check.

In [ ]:
import os
from tqdm.notebook import tnrange
from Crypto.Cipher import AES

project_file = "projects/Tutorial_AES_CWLITEXMEGA_fixedkey.cwp"
project = cw.create_project(project_file, overwrite=True)

N = 3000

fixed_key = bytearray([
    0x2B, 0x7E, 0x15, 0x16,
    0x28, 0xAE, 0xD2, 0xA6,
    0xAB, 0xF7, 0x15, 0x88,
    0x09, 0xCF, 0x4F, 0x3C
])

for i in tnrange(N, desc="Capturing traces"):
    text = bytearray(os.urandom(16))

    ret = cw.capture_trace(scope, target, text, fixed_key)

    if ret is None:
        print(f"Capture failed at trace {i}")
        continue

    cipher = AES.new(bytes(fixed_key), AES.MODE_ECB)
    expected = cipher.encrypt(bytes(text))

    assert bytes(ret.textout) == expected, (
        f"Incorrect encryption result on trace {i}\n"
        f"Got: {bytes(ret.textout).hex()}\n"
        f"Exp: {expected.hex()}"
    )

    project.traces.append(ret)

project.save()

print("Capture complete.")
print("Project saved to:", project_file)
print("Fixed key:", bytes(fixed_key).hex())
print("Number of traces stored:", len(project.traces))

This shows how a captured trace can be plotted:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

trace = np.array(project.traces[0].wave, dtype=float)

plt.figure(figsize=(16, 8))
plt.plot(trace)
plt.title("Sample XMEGA AES Trace")
plt.xlabel("Sample")
plt.ylabel("Amplitude")
plt.show()

Finally we save our traces and disconnect. By saving the traces, the attack can be repeated in the future without having to repeat the trace acquisition steps above.

## Attack
Now we re-open our saved project and specify the attack parameters. For this hardware AES implementation, we use a different leakage model and attack than what is used for the software AES implementations.

Note that this attack requires only the ciphertext, not the plaintext.

In [ ]:
import chipwhisperer.analyzer as cwa

project = cw.open_project("projects/Tutorial_AES_CWLITEXMEGA_fixedkey.cwp")

print("Project reopened.")
print("Trace count:", len(project.traces))

This runs the attack:

In [ ]:
attack = cwa.cpa(project, cwa.leakage_models.sbox_output)
cb = cwa.get_jupyter_callback(attack)

results = attack.run(cb)

The attack results can be saved for later viewing or processing without having to repeat the attack:

In [ ]:
import pickle

RESULT_PATH = "projects/Tutorial_AES_CWLITEXMEGA_fixedkey_results.pkl"

with open(RESULT_PATH, "wb") as f:
    pickle.dump(results, f)

print("Attack results saved to:", RESULT_PATH)

You may notice that we didn't get the expected key from this attack, but still got a good difference in correlation between the best guess and the next best guess. This is because we actually recovered the key from the last round of AES. We'll need to use analyzer to get the actual AES key: 

## Tests
Check that the key obtained by the attack is the key that was used.
This attack targets the last round key, so we have to roll it back to compare against the key we provided.

In [ ]:
recovered_key = [subkey[0][0] for subkey in results.find_maximums()]

print("Recovered key:")
print(" ".join(f"{b:02X}" for b in recovered_key))

## Next steps

The `jupyter/demos/CW305_ECC/` folder contains a series of tutorials for attacking hardware ECC on the CW305.

This CW305 appnote contains additional details on the CW305 platform: http://media.newae.com/appnotes/NAE0010_Whitepaper_CW305_AES_SCA_Attack.pdf

In [ ]:
expected_key = [
    0x2B, 0x7E, 0x15, 0x16,
    0x28, 0xAE, 0xD2, 0xA6,
    0xAB, 0xF7, 0x15, 0x88,
    0x09, 0xCF, 0x4F, 0x3C
]

print("=" * 60)
print("AES CPA ATTACK RESULT")
print("=" * 60)

print("Original key : ", " ".join(f"{b:02X}" for b in expected_key))
print("Recovered key: ", " ".join(f"{b:02X}" for b in recovered_key))
print()

print("Idx | Original | Recovered | Status")
print("-----------------------------------")

correct = 0
for i, (orig, rec) in enumerate(zip(expected_key, recovered_key)):
    status = "✅" if orig == rec else "❌"
    if orig == rec:
        correct += 1
    print(f"{i:>3} |   {orig:02X}    |    {rec:02X}    | {status}")

print()
print(f"Correct bytes: {correct}/16")

In [ ]:
target.dis()
scope.dis()
print("Disconnected")